# Frozen-Configuration Validation on MIMII Pump and Valve

This notebook evaluates the configuration selected using MIMII Fan on two additional
MIMII machine types. Pump and Valve were not used to select the feature representation,
detector, aggregation rule, threshold policy, split ratio, or seeds.

The experiment is deliberately frozen: six-band window features, training-only
standardisation, 20-nearest-neighbour distance, mean-of-ten clip aggregation, and a
validation-normal clip order-statistic threshold with strict `>` alerts.

Dataset locations are supplied explicitly through `MIMII_PUMP_6DB_ROOT` and
`MIMII_VALVE_6DB_ROOT`. Example values are `<pump-6db-root>` and
`<valve-6db-root>`; no common folder-naming convention is assumed.


In [1]:
import hashlib
import os
import platform
import time
import wave
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import signal
from scipy.io import wavfile
import sklearn
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "notebooks").is_dir() and (candidate / "reports").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the repository.")


REPO_ROOT = find_repo_root(Path.cwd())
REPORT_DIR = REPO_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ENV = {
    "pump": "MIMII_PUMP_6DB_ROOT",
    "valve": "MIMII_VALVE_6DB_ROOT",
}
missing_env = [name for name in DATASET_ENV.values() if not os.environ.get(name)]
if missing_env:
    raise RuntimeError(f"Set explicit dataset roots using: {', '.join(missing_env)}")
DATASET_ROOTS = {
    machine_type: Path(os.environ[env_name]).expanduser().resolve()
    for machine_type, env_name in DATASET_ENV.items()
}

MACHINE_TYPES = ("pump", "valve")
MACHINE_IDS = ("id_00", "id_02", "id_04", "id_06")
SEEDS = tuple(range(5))
TRAIN_FRACTION = 0.60
VALIDATION_FRACTION = 0.20
TARGET_VALIDATION_CLIP_FAR = 0.01

SAMPLE_RATE = 16_000
CLIP_SECONDS = 10
WINDOW_SECONDS = 1.0
WINDOWS_PER_CLIP = 10
NFFT = 1024
BAND_EDGES_HZ = [0, 250, 500, 1000, 2000, 4000, 8000]
BAND_LABELS = [f"{lo}-{hi}" for lo, hi in zip(BAND_EDGES_HZ, BAND_EDGES_HZ[1:])]
FEATURE_DIMENSION = 6
KNN_NEIGHBOURS = 20
DETECTOR_NAME = "knn_20"
AGGREGATION_NAME = "mean_10"
ALERT_OPERATOR = ">"

EXPECTED_COUNTS = {
    ("pump", "id_00"): (1006, 143, 603, 201, 202),
    ("pump", "id_02"): (1005, 111, 603, 201, 201),
    ("pump", "id_04"): (702, 100, 421, 140, 141),
    ("pump", "id_06"): (1036, 102, 621, 207, 208),
    ("valve", "id_00"): (991, 119, 594, 198, 199),
    ("valve", "id_02"): (708, 120, 424, 141, 143),
    ("valve", "id_04"): (1000, 120, 600, 200, 200),
    ("valve", "id_06"): (992, 120, 595, 198, 199),
}
EXPECTED_ALLOWED_FP = {
    ("pump", "id_00"): 2,
    ("pump", "id_02"): 2,
    ("pump", "id_04"): 1,
    ("pump", "id_06"): 2,
    ("valve", "id_00"): 1,
    ("valve", "id_02"): 1,
    ("valve", "id_04"): 2,
    ("valve", "id_06"): 1,
}

assert scipy.__version__ == "1.14.0"
assert BAND_EDGES_HZ == [0, 250, 500, 1000, 2000, 4000, 8000]
assert NFFT == 1024 and SAMPLE_RATE == 16_000 and WINDOW_SECONDS == 1.0
assert FEATURE_DIMENSION == len(BAND_LABELS) == 6
assert KNN_NEIGHBOURS == 20 and SEEDS == (0, 1, 2, 3, 4)
assert np.isclose(TRAIN_FRACTION + VALIDATION_FRACTION, 0.80)

print("Frozen configuration loaded.")
print(f"Python {platform.python_version()}, NumPy {np.__version__}, SciPy {scipy.__version__}, scikit-learn {sklearn.__version__}")
print("Dataset roots supplied explicitly by environment variables; absolute paths are not published.")


Frozen configuration loaded.
Python 3.12.4, NumPy 1.26.4, SciPy 1.14.0, scikit-learn 1.9.0
Dataset roots supplied explicitly by environment variables; absolute paths are not published.


## Frozen configuration

- Channel 0, 16 kHz, ten-second clips, ten non-overlapping one-second windows.
- `scipy.signal.spectrogram`, `nperseg=1024`, `noverlap=512`.
- Six bands: 0–250, 250–500, 500–1000, 1000–2000, 2000–4000,
  and 4000–8000 Hz.
- Band value: `10*log10(mean PSD + 1e-12)`.
- `StandardScaler` and 20-NN fit on one ID's normal training windows only.
- Window score: mean Euclidean distance to 20 nearest training windows.
- Clip score: arithmetic mean of its ten window scores (`mean_10`).
- Threshold: validation-normal clip order statistic allowing
  `floor(0.01*n_validation)` strict exceedances.


In [2]:
def load_clip(path: Path) -> tuple[int, np.ndarray]:
    '''Load channel 0 using the exact completed Fan EDA numeric conversion.'''
    rate, samples = wavfile.read(path)
    if samples.ndim > 1:
        samples = samples[:, 0]
    if np.issubdtype(samples.dtype, np.integer):
        samples = samples / np.iinfo(samples.dtype).max
    return rate, samples.astype(np.float32)


def windows(samples: np.ndarray, rate: int, seconds: float = WINDOW_SECONDS):
    n = int(rate * seconds)
    for index in range(len(samples) // n):
        yield index, samples[index * n : (index + 1) * n]


def band_energies(samples: np.ndarray, rate: int) -> np.ndarray:
    '''Mean energy per frozen band over one window, in dB.'''
    frequencies, _, power = signal.spectrogram(
        samples, fs=rate, nperseg=NFFT, noverlap=NFFT // 2
    )
    band_index = np.digitize(frequencies, BAND_EDGES_HZ[1:-1])
    return np.array([
        10 * np.log10(power[band_index == band].mean() + 1e-12)
        for band in range(len(BAND_LABELS))
    ])


def auroc(normal_scores, abnormal_scores) -> float:
    labels = np.r_[np.zeros(len(normal_scores)), np.ones(len(abnormal_scores))]
    scores = np.r_[normal_scores, abnormal_scores]
    return float(roc_auc_score(labels, scores))


def discover_clips():
    rows = []
    for machine_type in MACHINE_TYPES:
        root = DATASET_ROOTS[machine_type]
        assert root.is_dir(), f"Missing configured {machine_type} dataset root"
        assert {p.name for p in root.iterdir() if p.is_dir()} == set(MACHINE_IDS)
        assert not [p for p in root.iterdir() if p.is_file()]
        for machine_id in MACHINE_IDS:
            id_root = root / machine_id
            assert {p.name for p in id_root.iterdir() if p.is_dir()} == {"normal", "abnormal"}
            assert not [p for p in id_root.iterdir() if p.is_file()]
            for label in ("normal", "abnormal"):
                label_root = id_root / label
                paths = tuple(sorted(label_root.glob("*.wav")))
                assert paths
                assert not [p for p in label_root.iterdir() if p.is_file() and p.suffix.lower() != ".wav"]
                rows.extend(
                    {
                        "machine_type": machine_type,
                        "machine_id": machine_id,
                        "label": label,
                        "clip_name": path.name,
                        "path": path,
                    }
                    for path in paths
                )
    return pd.DataFrame(rows)


def split_normal_files(clips: pd.DataFrame, machine_type: str, machine_id: str, seed: int):
    selected = clips.query(
        "machine_type == @machine_type and machine_id == @machine_id and label == 'normal'"
    ).sort_values("path")
    paths = selected.path.to_numpy(dtype=object)
    order = np.random.default_rng(seed).permutation(len(paths))
    shuffled = paths[order]
    train_end = int(np.floor(TRAIN_FRACTION * len(shuffled)))
    validation_end = train_end + int(np.floor(VALIDATION_FRACTION * len(shuffled)))
    split = {
        "train": tuple(shuffled[:train_end]),
        "validation": tuple(shuffled[train_end:validation_end]),
        "test_normal": tuple(shuffled[validation_end:]),
    }
    split_sets = [set(values) for values in split.values()]
    assert all(split_sets[i].isdisjoint(split_sets[j]) for i in range(3) for j in range(i + 1, 3))
    assert set().union(*split_sets) == set(paths)
    assert split == split_normal_files_replay(clips, machine_type, machine_id, seed)
    for values in split.values():
        metadata = clips[clips.path.isin(values)]
        assert set(metadata.label) == {"normal"}
        assert set(metadata.machine_type) == {machine_type}
        assert set(metadata.machine_id) == {machine_id}
    return split


def split_normal_files_replay(clips, machine_type, machine_id, seed):
    selected = clips.query(
        "machine_type == @machine_type and machine_id == @machine_id and label == 'normal'"
    ).sort_values("path")
    paths = selected.path.to_numpy(dtype=object)
    shuffled = paths[np.random.default_rng(seed).permutation(len(paths))]
    train_end = int(np.floor(TRAIN_FRACTION * len(shuffled)))
    validation_end = train_end + int(np.floor(VALIDATION_FRACTION * len(shuffled)))
    return {
        "train": tuple(shuffled[:train_end]),
        "validation": tuple(shuffled[train_end:validation_end]),
        "test_normal": tuple(shuffled[validation_end:]),
    }


def split_fingerprint(split) -> str:
    payload = []
    for name in ("train", "validation", "test_normal"):
        payload.append(f"--{name}--")
        payload.extend(path.name + "|" + path.parent.parent.name for path in split[name])
    return hashlib.sha256("\n".join(payload).encode("utf-8")).hexdigest()


def extract_feature_cache(clips: pd.DataFrame):
    cache = {}
    format_rows = []
    ordered = clips.sort_values(["machine_type", "machine_id", "label", "path"])
    started = time.perf_counter()
    for number, row in enumerate(ordered.itertuples(index=False), start=1):
        with wave.open(str(row.path), "rb") as handle:
            header = {
                "rate": handle.getframerate(),
                "frames": handle.getnframes(),
                "channels": handle.getnchannels(),
                "sample_width": handle.getsampwidth(),
                "compression": handle.getcomptype(),
            }
        assert header == {
            "rate": SAMPLE_RATE,
            "frames": SAMPLE_RATE * CLIP_SECONDS,
            "channels": 8,
            "sample_width": 2,
            "compression": "NONE",
        }
        rate, samples = load_clip(row.path)
        assert rate == SAMPLE_RATE and len(samples) == SAMPLE_RATE * CLIP_SECONDS
        features = np.asarray(
            [band_energies(window, rate) for _, window in windows(samples, rate)],
            dtype=float,
        )
        assert features.shape == (WINDOWS_PER_CLIP, FEATURE_DIMENSION)
        assert np.isfinite(features).all()
        cache[row.path] = features
        format_rows.append({
            "machine_type": row.machine_type,
            "machine_id": row.machine_id,
            **header,
            "windows": len(features),
        })
        if number % 250 == 0 or number == len(ordered):
            elapsed = time.perf_counter() - started
            print(f"Extracted {number:>4}/{len(ordered)} clips ({elapsed:,.1f} s)", flush=True)
    return cache, pd.DataFrame(format_rows), time.perf_counter() - started


def stack_file_features(paths, feature_cache):
    values = np.vstack([feature_cache[path] for path in paths])
    assert values.shape == (WINDOWS_PER_CLIP * len(paths), FEATURE_DIMENSION)
    assert np.isfinite(values).all()
    return values


def fit_knn(machine_type, machine_id, train_paths, feature_cache, clips):
    metadata = clips[clips.path.isin(train_paths)]
    assert set(metadata.machine_type) == {machine_type}
    assert set(metadata.machine_id) == {machine_id}
    assert set(metadata.label) == {"normal"}
    train_features = stack_file_features(train_paths, feature_cache)
    started = time.perf_counter()
    scaler = StandardScaler().fit(train_features)
    scaled_train = scaler.transform(train_features)
    assert np.isfinite(scaler.mean_).all() and np.isfinite(scaler.scale_).all()
    assert np.all(scaler.scale_ > 0) and np.isfinite(scaled_train).all()
    model = NearestNeighbors(
        n_neighbors=KNN_NEIGHBOURS,
        metric="euclidean",
        algorithm="auto",
        n_jobs=1,
    ).fit(scaled_train)
    return {
        "machine_type": machine_type,
        "machine_id": machine_id,
        "scaler": scaler,
        "model": model,
        "fit_seconds": time.perf_counter() - started,
    }


def score_knn(model, machine_type, machine_id, features):
    assert model["machine_type"] == machine_type and model["machine_id"] == machine_id
    scaled = model["scaler"].transform(features)
    chunks = []
    for start in range(0, len(scaled), 512):
        distances = model["model"].kneighbors(
            scaled[start : start + 512], return_distance=True
        )[0]
        chunks.append(distances.mean(axis=1))
    scores = np.concatenate(chunks).astype(float)
    assert scores.shape == (len(features),) and np.isfinite(scores).all()
    return scores


def clip_mean_scores(window_scores, paths):
    assert len(window_scores) == WINDOWS_PER_CLIP * len(paths)
    values = np.asarray(window_scores, dtype=float).reshape(len(paths), WINDOWS_PER_CLIP)
    scores = values.mean(axis=1)
    assert np.isfinite(scores).all()
    return scores


def calibrate_clip_threshold(validation_clip_scores):
    scores = np.asarray(validation_clip_scores, dtype=float)
    assert scores.ndim == 1 and len(scores) > 0 and np.isfinite(scores).all()
    n = len(scores)
    allowed_fp = int(np.floor(TARGET_VALIDATION_CLIP_FAR * n))
    sorted_scores = np.sort(scores)
    threshold_index = n - allowed_fp - 1
    threshold = float(sorted_scores[threshold_index])
    actual_fp = int(np.sum(scores > threshold))
    ties = int(np.sum(scores == threshold))
    assert 0 <= actual_fp <= allowed_fp
    assert threshold == float(np.sort(scores.copy())[n - allowed_fp - 1])
    return {
        "validation_clip_count": n,
        "allowed_validation_false_positive_clips": allowed_fp,
        "threshold_order_statistic_one_based": n - allowed_fp,
        "threshold_zero_based_index": threshold_index,
        "clip_threshold": threshold,
        "actual_validation_false_positive_clips": actual_fp,
        "validation_threshold_tie_count": ties,
        "validation_clip_far": actual_fp / n,
        "alert_operator": ALERT_OPERATOR,
    }


In [3]:
clips = discover_clips()
inventory = (
    clips.groupby(["machine_type", "machine_id", "label"])
    .size().unstack(fill_value=0).reset_index()
)
inventory["total"] = inventory["normal"] + inventory["abnormal"]

for row in inventory.itertuples(index=False):
    normal, abnormal, *_ = EXPECTED_COUNTS[(row.machine_type, row.machine_id)]
    assert row.normal == normal and row.abnormal == abnormal
assert len(clips) == 8_375
assert set(clips.label) == {"normal", "abnormal"}
assert clips.path.is_unique

print("Dataset inventory and explicit-root assertions passed.")
display(inventory)


Dataset inventory and explicit-root assertions passed.


label,machine_type,machine_id,abnormal,normal,total
0,pump,id_00,143,1006,1149
1,pump,id_02,111,1005,1116
2,pump,id_04,100,702,802
3,pump,id_06,102,1036,1138
4,valve,id_00,119,991,1110
5,valve,id_02,120,708,828
6,valve,id_04,120,1000,1120
7,valve,id_06,120,992,1112


In [4]:
feature_cache, format_audit, feature_extraction_seconds = extract_feature_cache(clips)
assert len(feature_cache) == len(clips) == 8_375
assert set(format_audit.rate) == {16_000}
assert set(format_audit.frames) == {160_000}
assert set(format_audit.channels) == {8}
assert set(format_audit.sample_width) == {2}
assert set(format_audit.compression) == {"NONE"}
assert set(format_audit.windows) == {10}
print(f"Frozen six-band feature extraction complete in {feature_extraction_seconds:,.1f} seconds.")


Extracted  250/8375 clips (4.9 s)


Extracted  500/8375 clips (10.0 s)


Extracted  750/8375 clips (14.6 s)


Extracted 1000/8375 clips (18.9 s)


Extracted 1250/8375 clips (22.8 s)


...

Extracted 7750/8375 clips (97.3 s)


Extracted 8000/8375 clips (100.4 s)


Extracted 8250/8375 clips (103.7 s)


Extracted 8375/8375 clips (105.6 s)


Frozen six-band feature extraction complete in 105.6 seconds.


In [5]:
def evaluate_condition(machine_type, machine_id, seed):
    split = split_normal_files(clips, machine_type, machine_id, seed)
    normal_count, abnormal_count, expected_train, expected_validation, expected_test = (
        EXPECTED_COUNTS[(machine_type, machine_id)]
    )
    assert tuple(map(len, split.values())) == (
        expected_train, expected_validation, expected_test
    )
    abnormal_paths = tuple(
        clips.query(
            "machine_type == @machine_type and machine_id == @machine_id and label == 'abnormal'"
        ).sort_values("path").path
    )
    assert len(abnormal_paths) == abnormal_count
    assert set(abnormal_paths).isdisjoint(set().union(*map(set, split.values())))
    assert all(feature_cache[path].shape == (10, 6) for path in (*abnormal_paths, *split["train"], *split["validation"], *split["test_normal"]))

    model = fit_knn(machine_type, machine_id, split["train"], feature_cache, clips)
    validation_features = stack_file_features(split["validation"], feature_cache)
    test_normal_features = stack_file_features(split["test_normal"], feature_cache)
    abnormal_features = stack_file_features(abnormal_paths, feature_cache)

    score_started = time.perf_counter()
    validation_window_scores = score_knn(model, machine_type, machine_id, validation_features)
    test_normal_window_scores = score_knn(model, machine_type, machine_id, test_normal_features)
    abnormal_window_scores = score_knn(model, machine_type, machine_id, abnormal_features)
    score_seconds = time.perf_counter() - score_started
    assert np.array_equal(
        score_knn(model, machine_type, machine_id, validation_features[:32]),
        score_knn(model, machine_type, machine_id, validation_features[:32]),
    )

    validation_clip_scores = clip_mean_scores(validation_window_scores, split["validation"])
    test_normal_clip_scores = clip_mean_scores(test_normal_window_scores, split["test_normal"])
    abnormal_clip_scores = clip_mean_scores(abnormal_window_scores, abnormal_paths)
    threshold = calibrate_clip_threshold(validation_clip_scores)
    assert threshold["allowed_validation_false_positive_clips"] == EXPECTED_ALLOWED_FP[(machine_type, machine_id)]

    test_fp = int(np.sum(test_normal_clip_scores > threshold["clip_threshold"]))
    detected_abnormal = int(np.sum(abnormal_clip_scores > threshold["clip_threshold"]))
    row = {
        "condition_id": f"{machine_type}_{machine_id}_seed_{seed}",
        "machine_type": machine_type,
        "machine_id": machine_id,
        "seed": seed,
        "split_fingerprint_sha256": split_fingerprint(split),
        "window_auroc": auroc(test_normal_window_scores, abnormal_window_scores),
        "clip_auroc": auroc(test_normal_clip_scores, abnormal_clip_scores),
        "clip_threshold": threshold["clip_threshold"],
        "validation_clip_far": threshold["validation_clip_far"],
        "test_normal_clip_far": test_fp / len(test_normal_clip_scores),
        "abnormal_clip_detection": detected_abnormal / len(abnormal_clip_scores),
        "actual_validation_false_positive_clips": threshold["actual_validation_false_positive_clips"],
        "validation_threshold_tie_count": threshold["validation_threshold_tie_count"],
        "test_normal_false_positive_clips": test_fp,
        "detected_abnormal_clips": detected_abnormal,
        "train_normal_clips": len(split["train"]),
        "validation_normal_clips": len(split["validation"]),
        "test_normal_clips": len(split["test_normal"]),
        "abnormal_test_clips": len(abnormal_paths),
        "train_windows": len(split["train"]) * WINDOWS_PER_CLIP,
        "validation_windows": len(split["validation"]) * WINDOWS_PER_CLIP,
        "test_normal_windows": len(split["test_normal"]) * WINDOWS_PER_CLIP,
        "abnormal_test_windows": len(abnormal_paths) * WINDOWS_PER_CLIP,
        "fit_time_seconds": model["fit_seconds"],
        "score_time_seconds": score_seconds,
        "feature_dimension": FEATURE_DIMENSION,
        "feature": "six_band",
        "detector": DETECTOR_NAME,
        "aggregation": AGGREGATION_NAME,
        "alert_operator": ALERT_OPERATOR,
    }
    assert all(np.isfinite(value) for value in row.values() if isinstance(value, (float, np.floating)))
    threshold_row = {
        "condition_id": row["condition_id"],
        "machine_type": machine_type,
        "machine_id": machine_id,
        "seed": seed,
        **threshold,
    }
    return row, threshold_row


result_rows = []
threshold_rows = []
evaluation_started = time.perf_counter()
for machine_type in MACHINE_TYPES:
    for machine_id in MACHINE_IDS:
        for seed in SEEDS:
            result, threshold = evaluate_condition(machine_type, machine_id, seed)
            result_rows.append(result)
            threshold_rows.append(threshold)
            print(
                f"Completed {result['condition_id']}: clip AUROC={result['clip_auroc']:.4f}, "
                f"test FAR={result['test_normal_clip_far']:.4f}, detection={result['abnormal_clip_detection']:.4f}",
                flush=True,
            )

per_repeat = pd.DataFrame(result_rows).sort_values(
    ["machine_type", "machine_id", "seed"]
).reset_index(drop=True)
thresholds = pd.DataFrame(threshold_rows).sort_values(
    ["machine_type", "machine_id", "seed"]
).reset_index(drop=True)
evaluation_seconds = time.perf_counter() - evaluation_started

assert len(per_repeat) == len(thresholds) == 40
assert per_repeat.groupby(["machine_type", "machine_id"]).size().eq(5).all()
assert set(per_repeat.feature_dimension) == {6}
assert set(per_repeat.detector) == {"knn_20"}
assert set(per_repeat.aggregation) == {"mean_10"}
assert set(per_repeat.alert_operator) == {">"}
assert np.isfinite(per_repeat.select_dtypes(include=[np.number])).all().all()
assert np.isfinite(thresholds.select_dtypes(include=[np.number])).all().all()
print(f"All 40 frozen conditions completed in {evaluation_seconds:,.1f} seconds.")
display(per_repeat.groupby(["machine_type", "machine_id"])[
    ["window_auroc", "clip_auroc", "validation_clip_far", "test_normal_clip_far", "abnormal_clip_detection"]
].mean().round(4))


Completed pump_id_00_seed_0: clip AUROC=0.9902, test FAR=0.0099, detection=0.8811


Completed pump_id_00_seed_1: clip AUROC=0.9914, test FAR=0.0099, detection=0.8811


Completed pump_id_00_seed_2: clip AUROC=0.9909, test FAR=0.0050, detection=0.8741

Completed pump_id_00_seed_3: clip AUROC=0.9938, test FAR=0.0050, detection=0.8881


Completed pump_id_00_seed_4: clip AUROC=0.9875, test FAR=0.0396, detection=0.9161


...

Completed valve_id_06_seed_2: clip AUROC=0.7048, test FAR=0.0050, detection=0.0000


Completed valve_id_06_seed_3: clip AUROC=0.7174, test FAR=0.0101, detection=0.0250


Completed valve_id_06_seed_4: clip AUROC=0.6723, test FAR=0.0050, detection=0.0083


All 40 frozen conditions completed in 8.8 seconds.


window_auroc  clip_auroc  validation_clip_far  \
machine_type machine_id                                                  
pump         id_00             0.9767      0.9908               0.0100   
             id_02             0.8613      0.8876               0.0100   
             id_04             0.9436      0.9567               0.0071   
             id_06             0.9725      0.9892               0.0097   
valve        id_00             0.8394      0.8916               0.0051   
             id_02             0.7658      0.8202               0.0071   
             id_04             0.7812      0.8277               0.0100   
             id_06             0.6348      0.7002               0.0051   

                         test_normal_clip_far  abnormal_clip_detection  
machine_type machine_id                                                 
pump         id_00                     0.0139                   0.8881  
             id_02                     0.0149                   0.5081  
             id_04                     0.0199                   0.5600  
             id_06                     0.0115                   0.7686  
valve        id_00                     0.0080                   0.0403  
             id_02                     0.0070                   0.0233  
             id_04                     0.0050                   0.0083  
             id_06                     0.0090                   0.0250

In [6]:
SUMMARY_METRICS = (
    "window_auroc",
    "clip_auroc",
    "clip_threshold",
    "validation_clip_far",
    "test_normal_clip_far",
    "abnormal_clip_detection",
    "actual_validation_false_positive_clips",
    "validation_threshold_tie_count",
    "test_normal_false_positive_clips",
    "detected_abnormal_clips",
    "fit_time_seconds",
    "score_time_seconds",
)


def flatten_summary(grouped, summary_level):
    frame = grouped[list(SUMMARY_METRICS)].agg(["mean", "std", "min", "max"]).reset_index()
    frame.columns = [
        column if isinstance(column, str) else "_".join(part for part in column if part)
        for column in frame.columns
    ]
    frame = frame.rename(columns={c: c.replace("_std", "_sd") for c in frame.columns})
    frame.insert(0, "summary_level", summary_level)
    frame["seeds"] = len(SEEDS)
    return frame


id_summary = flatten_summary(
    per_repeat.groupby(["machine_type", "machine_id"]), "machine_id"
)
macro_by_seed = (
    per_repeat.groupby(["machine_type", "seed"])[list(SUMMARY_METRICS)]
    .mean().reset_index()
)
macro_by_seed["machine_id"] = "ALL_IDS_MACRO"
macro_summary = flatten_summary(
    macro_by_seed.groupby(["machine_type", "machine_id"]), "machine_type_macro"
)
summary = pd.concat([id_summary, macro_summary], ignore_index=True)

config_rows = [
    ("experiment", "frozen Pump/Valve within-MIMII validation", "task protocol"),
    ("dataset_root_pump", "environment variable MIMII_PUMP_6DB_ROOT", "portable runtime configuration"),
    ("dataset_root_valve", "environment variable MIMII_VALVE_6DB_ROOT", "portable runtime configuration"),
    ("channel", "0", "Fan-selected configuration"),
    ("sample_rate_hz", "16000", "Fan-selected configuration"),
    ("clip_seconds", "10", "audited dataset contract"),
    ("window_seconds", "1", "Fan-selected configuration"),
    ("window_overlap_seconds", "0", "Fan-selected configuration"),
    ("spectrogram_function", "scipy.signal.spectrogram", "Fan-selected configuration"),
    ("spectrogram_nperseg", "1024", "Fan-selected configuration"),
    ("spectrogram_noverlap", "512", "Fan-selected configuration"),
    ("band_edges_hz", "0,250,500,1000,2000,4000,8000", "Fan-selected configuration"),
    ("band_value", "10*log10(mean PSD + 1e-12)", "Fan-selected configuration"),
    ("feature_dimension", "6", "frozen"),
    ("scaler", "StandardScaler fitted on normal training windows only", "frozen"),
    ("detector", "NearestNeighbors", "frozen"),
    ("n_neighbors", "20", "frozen"),
    ("metric", "euclidean", "frozen"),
    ("algorithm", "auto", "frozen"),
    ("n_jobs", "1", "frozen"),
    ("window_score", "mean distance to 20 nearest training windows", "frozen"),
    ("clip_aggregation", "mean_10", "frozen"),
    ("normal_split", "floor(60%)/floor(20%)/remainder", "frozen"),
    ("seeds", "0,1,2,3,4", "frozen"),
    ("threshold", "validation-normal clip order statistic", "frozen"),
    ("target_validation_clip_far", "0.01", "frozen"),
    ("alert_operator", ">", "frozen"),
    ("source_research_notebook", "notebooks/eda_mimii_fan.ipynb", "provenance"),
    ("numpy_version", np.__version__, "execution environment"),
    ("scipy_version", scipy.__version__, "execution environment"),
    ("scikit_learn_version", sklearn.__version__, "execution environment"),
    ("feature_extraction_seconds", f"{feature_extraction_seconds:.6f}", "execution timing"),
    ("evaluation_seconds", f"{evaluation_seconds:.6f}", "execution timing"),
]
config = pd.DataFrame(config_rows, columns=["parameter", "value", "provenance"])

per_repeat.to_csv(REPORT_DIR / "pump_valve_frozen_validation_per_repeat.csv", index=False)
summary.to_csv(REPORT_DIR / "pump_valve_frozen_validation_summary.csv", index=False)
thresholds.to_csv(REPORT_DIR / "pump_valve_frozen_validation_thresholds.csv", index=False)
config.to_csv(REPORT_DIR / "pump_valve_frozen_validation_config.csv", index=False)

assert len(id_summary) == 8 and len(macro_summary) == 2 and len(summary) == 10
display(summary[[
    "summary_level", "machine_type", "machine_id",
    "clip_auroc_mean", "clip_auroc_sd",
    "test_normal_clip_far_mean", "abnormal_clip_detection_mean",
]].round(4))


,summary_level,machine_type,machine_id,clip_auroc_mean,clip_auroc_sd,test_normal_clip_far_mean,abnormal_clip_detection_mean
0,machine_id,pump,id_00,0.9908,0.0023,0.0139,0.8881
1,machine_id,pump,id_02,0.8876,0.0115,0.0149,0.5081
2,machine_id,pump,id_04,0.9567,0.0065,0.0199,0.5600
3,machine_id,pump,id_06,0.9892,0.0058,0.0115,0.7686
4,machine_id,valve,id_00,0.8916,0.0060,0.0080,0.0403
5,machine_id,valve,id_02,0.8202,0.0248,0.0070,0.0233
6,machine_id,valve,id_04,0.8277,0.0070,0.0050,0.0083
7,machine_id,valve,id_06,0.7002,0.0175,0.0090,0.0250
8,machine_type_macro,pump,ALL_IDS_MACRO,0.9561,0.0057,0.0150,0.6812
9,machine_type_macro,valve,ALL_IDS_MACRO,0.8099,0.0081,0.0073,0.0243


In [7]:
plot_data = id_summary.sort_values(["machine_type", "machine_id"]).reset_index(drop=True)
labels = [f"{row.machine_type.title()} {row.machine_id}" for row in plot_data.itertuples()]
colours = ["#1f77b4" if value == "pump" else "#d62728" for value in plot_data.machine_type]

fig, ax = plt.subplots(figsize=(11, 5.5))
x = np.arange(len(plot_data))
ax.errorbar(
    x,
    plot_data.clip_auroc_mean,
    yerr=plot_data.clip_auroc_sd,
    fmt="none",
    ecolor="0.35",
    elinewidth=1.6,
    capsize=4,
)
ax.scatter(x, plot_data.clip_auroc_mean, c=colours, s=58, zorder=3)
ax.axhline(0.9, color="0.35", linestyle="--", linewidth=1.2, label="Historical 0.9 reference")
ax.set_xticks(x, labels, rotation=28, ha="right")
ax.set_ylim(0, 1.03)
ax.set_ylabel("Clip AUROC (mean ± sample SD across seeds)")
ax.set_title("Frozen six-band 20-NN + mean_10 clip ranking")
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(REPORT_DIR / "pump_valve_frozen_validation_clip_auroc.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
markers = {"pump": "o", "valve": "s"}
palette = {"pump": "#1f77b4", "valve": "#d62728"}
annotation_offsets = {
    ("valve", "id_00"): (8, 10),
    ("valve", "id_02"): (-38, -13),
    ("valve", "id_04"): (8, 5),
    ("valve", "id_06"): (8, 9),
}
for machine_type in MACHINE_TYPES:
    subset = plot_data.query("machine_type == @machine_type")
    ax.errorbar(
        subset.test_normal_clip_far_mean,
        subset.abnormal_clip_detection_mean,
        xerr=subset.test_normal_clip_far_sd,
        yerr=subset.abnormal_clip_detection_sd,
        fmt=markers[machine_type],
        color=palette[machine_type],
        ecolor=palette[machine_type],
        capsize=3,
        markersize=7,
        linestyle="none",
        label=machine_type.title(),
    )
    for row in subset.itertuples():
        offset = annotation_offsets.get((machine_type, row.machine_id), (5, 4))
        ax.annotate(row.machine_id, (row.test_normal_clip_far_mean, row.abnormal_clip_detection_mean), xytext=offset, textcoords="offset points", fontsize=8)
ax.axvline(0.01, color="0.35", linestyle="--", linewidth=1.2, label="1% clip FAR reference")
ax.set_xlim(left=0)
ax.set_ylim(0, 1.03)
ax.set_xlabel("Test-normal clip FAR (mean ± sample SD)")
ax.set_ylabel("Abnormal-clip detection (mean ± sample SD)")
ax.set_title("Frozen clip operating point by machine ID")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(REPORT_DIR / "pump_valve_frozen_validation_far_detection.png", dpi=200, bbox_inches="tight")
plt.show()


D:\Temp\ipykernel_8512\215616408.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


D:\Temp\ipykernel_8512\215616408.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
def markdown_table(frame):
    values = frame.astype(str)
    header = "| " + " | ".join(values.columns) + " |"
    rule = "| " + " | ".join("---" for _ in values.columns) + " |"
    rows = ["| " + " | ".join(row) + " |" for row in values.to_numpy()]
    return "\n".join([header, rule, *rows])


def result_table(machine_type):
    frame = id_summary.query("machine_type == @machine_type").copy()
    return pd.DataFrame({
        "ID": frame.machine_id,
        "window AUROC": frame.apply(lambda r: f"{r.window_auroc_mean:.4f} ± {r.window_auroc_sd:.4f}", axis=1),
        "clip AUROC": frame.apply(lambda r: f"{r.clip_auroc_mean:.4f} ± {r.clip_auroc_sd:.4f}", axis=1),
        "validation clip FAR": frame.apply(lambda r: f"{r.validation_clip_far_mean:.4%} ± {r.validation_clip_far_sd:.4%}", axis=1),
        "test-normal clip FAR": frame.apply(lambda r: f"{r.test_normal_clip_far_mean:.4%} ± {r.test_normal_clip_far_sd:.4%}", axis=1),
        "abnormal-clip detection": frame.apply(lambda r: f"{r.abnormal_clip_detection_mean:.4%} ± {r.abnormal_clip_detection_sd:.4%}", axis=1),
    })


def stability_table():
    frame = id_summary.copy()
    return pd.DataFrame({
        "Type / ID": frame.machine_type.str.title() + " " + frame.machine_id,
        "clip AUROC min–max": frame.apply(lambda r: f"{r.clip_auroc_min:.4f}–{r.clip_auroc_max:.4f}", axis=1),
        "test-normal clip FAR min–max": frame.apply(lambda r: f"{r.test_normal_clip_far_min:.4%}–{r.test_normal_clip_far_max:.4%}", axis=1),
        "detection min–max": frame.apply(lambda r: f"{r.abnormal_clip_detection_min:.4%}–{r.abnormal_clip_detection_max:.4%}", axis=1),
    })


id_values = id_summary.copy()
ranking_above_reference = int((id_values.clip_auroc_mean >= 0.9).sum())
far_above_one_percent = int((id_values.test_normal_clip_far_mean > 0.01).sum())
clip_span = float(id_values.clip_auroc_mean.max() - id_values.clip_auroc_mean.min())
detection_span = float(id_values.abnormal_clip_detection_mean.max() - id_values.abnormal_clip_detection_mean.min())
best = id_values.loc[id_values.clip_auroc_mean.idxmax()]
weakest = id_values.loc[id_values.clip_auroc_mean.idxmin()]

if clip_span >= 0.15 or detection_span >= 0.50:
    outcome = "Outcome C — machine-ID-specific transfer"
    outcome_text = (
        "The frozen approach transfers unevenly: the spread between machine IDs is too large "
        "to describe performance with a single pooled conclusion."
    )
elif ranking_above_reference >= 6 and far_above_one_percent >= 4:
    outcome = "Outcome B — ranking transfers better than calibration"
    outcome_text = (
        "Clip ranking is generally strong, but the frozen clip operating point does not "
        "consistently preserve low false-alert rates on held-out normal clips."
    )
elif ranking_above_reference >= 6 and far_above_one_percent < 4:
    outcome = "Outcome A — broad within-MIMII transfer"
    outcome_text = (
        "Most IDs retain strong ranking and comparatively stable clip-level operation, "
        "supporting transfer across these additional MIMII machine types."
    )
else:
    outcome = "Outcome D — poor transfer"
    outcome_text = (
        "Many IDs lose ranking or useful alerting behaviour, so the Fan-selected "
        "configuration does not transfer reliably across the evaluated machine types."
    )

finding_lines = []
for row in id_values.sort_values(["machine_type", "machine_id"]).itertuples():
    finding_lines.append(
        f"- **{row.machine_type.title()} {row.machine_id}:** mean clip AUROC "
        f"{row.clip_auroc_mean:.4f}; mean test-normal clip FAR "
        f"{row.test_normal_clip_far_mean:.2%}; mean abnormal-clip detection "
        f"{row.abnormal_clip_detection_mean:.2%}."
    )

pump_table = markdown_table(result_table("pump"))
valve_table = markdown_table(result_table("valve"))
stability = markdown_table(stability_table())

report = f'''# Frozen-Configuration Validation on MIMII Pump and Valve

## 1. Motivation

This experiment tests whether the configuration selected using MIMII Fan transfers to
Pump and Valve without retrospective tuning. Pump and Valve were not used in the Fan
feature, detector, aggregation, or threshold-policy selection experiments. This is
within-MIMII transfer evidence, not external-domain validation.

## 2. Frozen Configuration

The frozen chain is channel 0, 16 kHz, ten one-second non-overlapping windows, six
fixed frequency bands, training-only `StandardScaler`, Euclidean 20-NN mean distance,
`mean_10` clip aggregation, and a validation-normal clip order-statistic threshold.
Alerts use strict `score > threshold`. Each machine ID has an independent scaler,
neighbour index, and threshold.

## 3. Dataset and Integrity

The audit covered 4,205 Pump and 4,170 Valve clips across IDs `id_00`, `id_02`,
`id_04`, and `id_06`. All 8,375 files were readable 16 kHz, ten-second, eight-channel,
16-bit PCM WAVs; channel 0 was available and every clip produced exactly ten windows.
The acquisition audit found no zero-byte files, unreadable files, or exact SHA-256
duplicates within IDs, across IDs of the same type, or across normal/abnormal labels.

## 4. Split and Threshold Protocol

Normal files were split before windowing using deterministic seeds 0–4: floor(60%)
training, floor(20%) validation, and the remainder normal test. All abnormal files were
test-only. The clip threshold was the validation-normal order statistic at index
`n - floor(0.01*n) - 1`; strict `>` means ties at the threshold do not alert.
Validation clip FAR therefore cannot exceed 1% empirically, but its resolution is
coarse where only one validation false positive is permitted.

## 5. Results

### 5.1 Pump

{pump_table}

### 5.2 Valve

{valve_table}

### 5.3 Clip Ranking

{ranking_above_reference} of 8 IDs have mean clip AUROC at or above the historical
0.9 reference. This line is context only, not a new universal success criterion.
The strongest mean ranking was {best.machine_type.title()} {best.machine_id}
({best.clip_auroc_mean:.4f}); the weakest was {weakest.machine_type.title()}
{weakest.machine_id} ({weakest.clip_auroc_mean:.4f}).

![Clip AUROC by machine ID](pump_valve_frozen_validation_clip_auroc.png)

### 5.4 Clip FAR and Abnormal-Clip Detection

{far_above_one_percent} of 8 IDs have mean held-out test-normal clip FAR above 1%.
Validation clip FAR at or below 1% does not guarantee test-normal clip FAR at or below
1%. Ranking and operational alerting are therefore reported separately.

![Clip FAR versus abnormal-clip detection](pump_valve_frozen_validation_far_detection.png)

### 5.5 Across-Seed Stability

{stability}

## 6. Machine-ID-Specific Findings

{chr(10).join(finding_lines)}

## 7. Comparison with Fan Context

Historical Fan mean clip AUROC values for the same six-band 20-NN + `mean_10`
configuration were approximately 0.8983 (`id_00`), 0.9934 (`id_02`), 0.9734
(`id_04`), and 1.0000 (`id_06`). They provide context only; Fan, Pump, and Valve
were not pooled and the configuration was not reselected.

## 8. Interpretation

**{outcome}.** {outcome_text} This conclusion concerns only the evaluated MIMII
machine types and does not establish external-domain generalisation.

## 9. Relationship to Supervisor Feedback

The experiment extends evaluation beyond Fan while holding the selected method fixed,
reports each machine ID rather than hiding weak cases in a pooled score, and separates
ranking quality from clip-level false-alert and detection behaviour. It therefore tests
transfer directly without converting the validation set into another tuning stage.

## 10. Limitations

- Pump and Valve share the MIMII acquisition domain; this is not external validation.
- Validation clip thresholds have coarse finite-sample resolution of one or two clips.
- Five deterministic splits quantify split sensitivity but not all deployment variation.
- Validation clip FAR at or below 1% does not guarantee held-out clip FAR at or below 1%.
- Results are research validation, not production validation.
- No post-result tuning is permitted inside this frozen experiment.

## 11. Conclusion

The Fan-selected configuration was evaluated unchanged across 40 Pump/Valve
conditions. The evidence supports **{outcome.lower()}**. Pump and Valve results must
remain per-ID evidence, with equal-ID macro summaries only as supplementary context.
DCASE remains optional future work for a genuinely different-domain study; it is not
required to interpret this within-MIMII transfer experiment.
'''

(REPORT_DIR / "pump_valve_frozen_validation.md").write_text(report, encoding="utf-8")
print(outcome)
print(outcome_text)


Outcome C — machine-ID-specific transfer
The frozen approach transfers unevenly: the spread between machine IDs is too large to describe performance with a single pooled conclusion.


In [9]:
# Final integrity and output validation.
required_outputs = [
    REPORT_DIR / "pump_valve_frozen_validation.md",
    REPORT_DIR / "pump_valve_frozen_validation_per_repeat.csv",
    REPORT_DIR / "pump_valve_frozen_validation_summary.csv",
    REPORT_DIR / "pump_valve_frozen_validation_thresholds.csv",
    REPORT_DIR / "pump_valve_frozen_validation_config.csv",
    REPORT_DIR / "pump_valve_frozen_validation_clip_auroc.png",
    REPORT_DIR / "pump_valve_frozen_validation_far_detection.png",
]
assert all(path.is_file() and path.stat().st_size > 0 for path in required_outputs)
assert len(per_repeat) == 40
assert set(per_repeat.groupby(["machine_type", "machine_id"]).size()) == {5}
assert per_repeat.groupby(["machine_type", "machine_id", "seed"]).size().eq(1).all()
assert np.isfinite(per_repeat.select_dtypes(include=[np.number])).all().all()
assert np.isfinite(summary.select_dtypes(include=[np.number])).all().all()
assert np.isfinite(thresholds.select_dtypes(include=[np.number])).all().all()
assert (thresholds.alert_operator == ">").all()
assert (
    thresholds.actual_validation_false_positive_clips
    <= thresholds.allowed_validation_false_positive_clips
).all()

saved_repeat = pd.read_csv(REPORT_DIR / "pump_valve_frozen_validation_per_repeat.csv")
saved_summary = pd.read_csv(REPORT_DIR / "pump_valve_frozen_validation_summary.csv")
saved_thresholds = pd.read_csv(REPORT_DIR / "pump_valve_frozen_validation_thresholds.csv")
pd.testing.assert_frame_equal(saved_repeat, per_repeat, check_dtype=False, rtol=1e-12, atol=1e-12)
pd.testing.assert_frame_equal(saved_summary, summary, check_dtype=False, rtol=1e-12, atol=1e-12)
pd.testing.assert_frame_equal(saved_thresholds, thresholds, check_dtype=False, rtol=1e-12, atol=1e-12)

published_text = "\n".join(
    path.read_text(encoding="utf-8", errors="ignore")
    for path in required_outputs
    if path.suffix in {".md", ".csv"}
)
assert "C:\\Users\\" not in published_text
assert "D:\\Datasets\\" not in published_text

print("All 8 IDs and 40 conditions completed.")
print("All frozen-configuration, leakage, finiteness, portability, and output assertions passed.")


All 8 IDs and 40 conditions completed.
All frozen-configuration, leakage, finiteness, portability, and output assertions passed.


# Valve Low-FAR Diagnostic

This final, narrow diagnostic is explanatory and post-hoc. The exact per-clip score
vectors were not serialized by the frozen validation, so they are regenerated by
re-running that pipeline exactly as written and reusing its existing splits, cached
six-band features, per-ID scaler and 20-NN fits, `mean_10` aggregation, and frozen
validation-normal threshold rule. No feature, model, aggregation, split, or threshold
policy is changed.

The post-hoc test-normal 1% threshold is used only to distinguish validation-to-test
threshold shift from weak extreme-tail separation. It is not a deployable operating
point and does not replace the frozen validation-selected threshold.


In [6]:
from sklearn.metrics import roc_curve

VALVE_DIAGNOSTIC_CSV = REPORT_DIR / "valve_low_far_diagnostic.csv"
VALVE_DIAGNOSTIC_FIGURE = REPORT_DIR / "valve_low_far_diagnostic.png"
VALVE_DIAGNOSTIC_REPORT = REPORT_DIR / "valve_low_far_diagnostic.md"
LOW_FPR_LIMIT = 0.01


def diagnostic_score_summary(prefix, values):
    values = np.asarray(values, dtype=float)
    assert values.ndim == 1 and len(values) > 0 and np.isfinite(values).all()
    quantiles = np.quantile(values, [0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    return {
        f"{prefix}_count": len(values),
        f"{prefix}_min": float(values.min()),
        f"{prefix}_p25": float(quantiles[0]),
        f"{prefix}_p50": float(quantiles[1]),
        f"{prefix}_p75": float(quantiles[2]),
        f"{prefix}_p90": float(quantiles[3]),
        f"{prefix}_p95": float(quantiles[4]),
        f"{prefix}_p99": float(quantiles[5]),
        f"{prefix}_max": float(values.max()),
    }


def empirical_low_fpr_point(fpr, tpr, roc_thresholds):
    eligible = np.flatnonzero(fpr <= LOW_FPR_LIMIT + 1e-15)
    assert len(eligible) > 0
    maximum_tpr = float(tpr[eligible].max())
    max_tpr_candidates = eligible[np.isclose(tpr[eligible], maximum_tpr, rtol=0, atol=1e-15)]
    max_tpr_index = int(max_tpr_candidates[np.argmax(fpr[max_tpr_candidates])])

    largest_fpr = float(fpr[eligible].max())
    largest_fpr_candidates = eligible[np.isclose(fpr[eligible], largest_fpr, rtol=0, atol=1e-15)]
    largest_fpr_index = int(largest_fpr_candidates[np.argmax(tpr[largest_fpr_candidates])])
    return {
        "max_tpr_at_fpr_le_0_01": maximum_tpr,
        "empirical_fpr_at_max_tpr_le_0_01": float(fpr[max_tpr_index]),
        "empirical_threshold_at_max_tpr_le_0_01": float(roc_thresholds[max_tpr_index]),
        "largest_empirical_fpr_le_0_01": largest_fpr,
        "tpr_at_largest_empirical_fpr_le_0_01": float(tpr[largest_fpr_index]),
        "threshold_at_largest_empirical_fpr_le_0_01": float(roc_thresholds[largest_fpr_index]),
    }


def recover_valve_clip_scores(machine_id, seed):
    machine_type = "valve"
    split = split_normal_files(clips, machine_type, machine_id, seed)
    abnormal_paths = tuple(
        clips.query(
            "machine_type == @machine_type and machine_id == @machine_id and label == 'abnormal'"
        ).sort_values("path").path
    )
    assert set(abnormal_paths).isdisjoint(set().union(*map(set, split.values())))
    model = fit_knn(machine_type, machine_id, split["train"], feature_cache, clips)
    assert model["machine_type"] == "valve" and model["machine_id"] == machine_id

    validation_clip_scores = clip_mean_scores(
        score_knn(
            model, machine_type, machine_id,
            stack_file_features(split["validation"], feature_cache),
        ),
        split["validation"],
    )
    test_normal_clip_scores = clip_mean_scores(
        score_knn(
            model, machine_type, machine_id,
            stack_file_features(split["test_normal"], feature_cache),
        ),
        split["test_normal"],
    )
    abnormal_clip_scores = clip_mean_scores(
        score_knn(
            model, machine_type, machine_id,
            stack_file_features(abnormal_paths, feature_cache),
        ),
        abnormal_paths,
    )
    assert all(np.isfinite(values).all() for values in (
        validation_clip_scores, test_normal_clip_scores, abnormal_clip_scores
    ))
    return split, validation_clip_scores, test_normal_clip_scores, abnormal_clip_scores


diagnostic_rows = []
diagnostic_roc = {}
for machine_id in MACHINE_IDS:
    for seed in SEEDS:
        split, validation_scores, test_normal_scores, abnormal_scores = (
            recover_valve_clip_scores(machine_id, seed)
        )
        frozen = calibrate_clip_threshold(validation_scores)
        frozen_threshold = frozen["clip_threshold"]
        frozen_far = float(np.mean(test_normal_scores > frozen_threshold))
        frozen_detection = float(np.mean(abnormal_scores > frozen_threshold))

        reference = per_repeat.query(
            "machine_type == 'valve' and machine_id == @machine_id and seed == @seed"
        )
        assert len(reference) == 1
        reference = reference.iloc[0]
        assert np.isclose(frozen_threshold, reference.clip_threshold, rtol=0, atol=1e-12)
        assert np.isclose(frozen_far, reference.test_normal_clip_far, rtol=0, atol=1e-12)
        assert np.isclose(frozen_detection, reference.abnormal_clip_detection, rtol=0, atol=1e-12)
        assert len(validation_scores) == reference.validation_normal_clips
        assert len(test_normal_scores) == reference.test_normal_clips
        assert len(abnormal_scores) == reference.abnormal_test_clips

        labels = np.r_[np.zeros(len(test_normal_scores)), np.ones(len(abnormal_scores))]
        scores = np.r_[test_normal_scores, abnormal_scores]
        fpr, tpr, roc_thresholds = roc_curve(labels, scores, drop_intermediate=False)
        low_fpr = empirical_low_fpr_point(fpr, tpr, roc_thresholds)
        standardized_partial_auc = float(
            roc_auc_score(labels, scores, max_fpr=LOW_FPR_LIMIT)
        )

        posthoc = calibrate_clip_threshold(test_normal_scores)
        posthoc_threshold = posthoc["clip_threshold"]
        posthoc_far = float(np.mean(test_normal_scores > posthoc_threshold))
        posthoc_tpr = float(np.mean(abnormal_scores > posthoc_threshold))
        assert posthoc_far <= LOW_FPR_LIMIT + 1e-15

        row = {
            "machine_id": machine_id,
            "seed": seed,
            "validation_threshold": frozen_threshold,
            "validation_clip_count": len(validation_scores),
            "test_normal_clip_count": len(test_normal_scores),
            "abnormal_clip_count": len(abnormal_scores),
            "frozen_test_normal_clip_far": frozen_far,
            "frozen_test_normal_false_positive_count": int(np.sum(test_normal_scores > frozen_threshold)),
            "frozen_abnormal_clip_detection": frozen_detection,
            "frozen_detected_abnormal_clip_count": int(np.sum(abnormal_scores > frozen_threshold)),
            **diagnostic_score_summary("validation_normal", validation_scores),
            **diagnostic_score_summary("test_normal", test_normal_scores),
            **diagnostic_score_summary("abnormal", abnormal_scores),
            "standardized_partial_auroc_fpr_0_0_01": standardized_partial_auc,
            **low_fpr,
            "posthoc_test_normal_1pct_threshold": posthoc_threshold,
            "posthoc_test_normal_1pct_far": posthoc_far,
            "posthoc_test_normal_false_positive_count": int(np.sum(test_normal_scores > posthoc_threshold)),
            "posthoc_abnormal_tpr": posthoc_tpr,
            "posthoc_detected_abnormal_clip_count": int(np.sum(abnormal_scores > posthoc_threshold)),
            "allowed_test_normal_fp_count": posthoc["allowed_validation_false_positive_clips"],
            "posthoc_test_normal_threshold_tie_count": posthoc["validation_threshold_tie_count"],
        }
        assert all(np.isfinite(value) for value in row.values() if isinstance(value, (float, np.floating)))
        diagnostic_rows.append(row)
        diagnostic_roc[(machine_id, seed)] = (fpr, tpr)


valve_diagnostic = pd.DataFrame(diagnostic_rows).sort_values(
    ["machine_id", "seed"]
).reset_index(drop=True)
assert len(valve_diagnostic) == 20
assert valve_diagnostic.groupby(["machine_id", "seed"]).size().eq(1).all()
assert set(valve_diagnostic.machine_id) == set(MACHINE_IDS)
assert set(valve_diagnostic.seed) == set(SEEDS)
assert np.isfinite(valve_diagnostic.select_dtypes(include=[np.number])).all().all()

# One CSV only: it contains per-seed counts, full cohort summaries, frozen results,
# low-FPR separability diagnostics, and explicitly post-hoc threshold sensitivity.
valve_diagnostic.to_csv(VALVE_DIAGNOSTIC_CSV, index=False)


diagnostic_y_max = min(1.0, max(0.20, 1.15 * valve_diagnostic[[
    "max_tpr_at_fpr_le_0_01", "frozen_abnormal_clip_detection", "posthoc_abnormal_tpr"
]].to_numpy().max()))
fig, axes = plt.subplots(2, 2, figsize=(11.0, 8.2), sharex=True, sharey=True)
for axis, machine_id in zip(axes.flat, MACHINE_IDS):
    selected = valve_diagnostic.query("machine_id == @machine_id").sort_values("seed")
    for row in selected.itertuples(index=False):
        fpr, tpr = diagnostic_roc[(machine_id, row.seed)]
        mask = fpr <= LOW_FPR_LIMIT + 1e-15
        axis.step(
            fpr[mask], tpr[mask], where="post", color="#4C78A8", alpha=0.35,
            linewidth=1.2, label="seed ROC" if row.seed == 0 else None,
        )
        frozen_x = min(row.frozen_test_normal_clip_far, LOW_FPR_LIMIT)
        frozen_marker = ">" if row.frozen_test_normal_clip_far > LOW_FPR_LIMIT else "o"
        axis.scatter(
            frozen_x, row.frozen_abnormal_clip_detection, marker=frozen_marker,
            s=38, facecolors="none", edgecolors="#D62728", linewidths=1.2,
            label="frozen point" if row.seed == 0 else None, zorder=4,
        )
        axis.scatter(
            row.posthoc_test_normal_1pct_far, row.posthoc_abnormal_tpr,
            marker="D", s=25, color="#111111", alpha=0.75,
            label="post-hoc 1% point" if row.seed == 0 else None, zorder=4,
        )
    axis.set_title(f"Valve {machine_id}")
    axis.set_xlim(0, LOW_FPR_LIMIT)
    axis.set_ylim(0, diagnostic_y_max)
    axis.grid(alpha=0.22)
    axis.set_xlabel("Held-out test-normal FPR")
    axis.set_ylabel("Abnormal clip TPR")

handles, labels_for_legend = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels_for_legend, loc="lower center", bbox_to_anchor=(0.5, 0.055), ncol=3, frameon=False)
fig.suptitle("Valve low-FPR ROC (frozen scores; post-hoc diagnostic only)", y=0.98)
fig.text(
    0.5, 0.018,
    "Right triangles at the 1% boundary denote frozen FAR values above 1%; "
    "diamonds are non-deployable test-normal-derived diagnostic points.",
    ha="center", fontsize=8.5,
)
fig.tight_layout(rect=(0, 0.12, 1, 0.95))
fig.savefig(VALVE_DIAGNOSTIC_FIGURE, dpi=180, bbox_inches="tight")
plt.close(fig)


def diagnostic_markdown_table(frame):
    values = frame.astype(str)
    header = "| " + " | ".join(values.columns) + " |"
    rule = "| " + " | ".join("---" for _ in values.columns) + " |"
    rows = ["| " + " | ".join(row) + " |" for row in values.to_numpy()]
    return "\n".join([header, rule, *rows])


summary_rows = []
interpretation_lines = []
for machine_id, group in valve_diagnostic.groupby("machine_id", sort=True):
    delta = group.posthoc_abnormal_tpr - group.frozen_abnormal_clip_detection
    # These labels summarize, but do not select or tune, the experiment. "Substantial"
    # is conservatively operationalized as >=10 percentage points in at least 3/5 seeds.
    substantial_seeds = int((delta >= 0.10).sum())
    if group.posthoc_abnormal_tpr.mean() <= 0.10 and substantial_seeds < 3:
        category = "B: tail-overlap limitation"
        explanation = (
            f"mean post-hoc TPR was {group.posthoc_abnormal_tpr.mean():.2%}, and "
            f"a >=10-point gain occurred in only {substantial_seeds}/5 seeds; "
            "conditioning on the held-out normal tail did not recover consistently useful separation"
        )
    elif substantial_seeds >= 3:
        category = "A: threshold-transfer contribution"
        explanation = (
            f"post-hoc TPR improved by at least 10 percentage points in "
            f"{substantial_seeds}/5 seeds"
        )
    else:
        category = "C: mixed mechanism"
        explanation = (
            "post-hoc changes were not consistently large and low-FAR separation "
            "remained machine-ID/seed dependent"
        )
    interpretation_lines.append(f"- **{machine_id}: {category}.** {explanation}.")
    summary_rows.append({
        "ID": machine_id,
        "frozen threshold mean": f"{group.validation_threshold.mean():.4f}",
        "frozen FAR mean (range)": (
            f"{group.frozen_test_normal_clip_far.mean():.2%} "
            f"({group.frozen_test_normal_clip_far.min():.2%}-{group.frozen_test_normal_clip_far.max():.2%})"
        ),
        "frozen detection mean (range)": (
            f"{group.frozen_abnormal_clip_detection.mean():.2%} "
            f"({group.frozen_abnormal_clip_detection.min():.2%}-{group.frozen_abnormal_clip_detection.max():.2%})"
        ),
        "max TPR <=1% FPR mean (range)": (
            f"{group.max_tpr_at_fpr_le_0_01.mean():.2%} "
            f"({group.max_tpr_at_fpr_le_0_01.min():.2%}-{group.max_tpr_at_fpr_le_0_01.max():.2%})"
        ),
        "post-hoc TPR mean (range)": (
            f"{group.posthoc_abnormal_tpr.mean():.2%} "
            f"({group.posthoc_abnormal_tpr.min():.2%}-{group.posthoc_abnormal_tpr.max():.2%})"
        ),
        "interpretation": category,
    })

diagnostic_summary = pd.DataFrame(summary_rows)
categories = set(diagnostic_summary.interpretation)
if categories == {"B: tail-overlap limitation"}:
    cross_valve = (
        "Diagnostic B is supported across all four Valve IDs: even after conditioning "
        "post-hoc on each held-out normal tail, abnormal clips rarely reached the "
        "extreme low-FPR region. Weak tail separation under the frozen score function "
        "is therefore the dominant observed limitation."
    )
elif categories == {"A: threshold-transfer contribution"}:
    cross_valve = (
        "Diagnostic A is supported across all four Valve IDs: validation-to-test score "
        "shift contributes materially to weak frozen detection, although this post-hoc "
        "result is not a replacement threshold or deployable estimate."
    )
else:
    cross_valve = (
        "Diagnostic C is supported across Valve overall: the relative roles of "
        "validation-to-test threshold shift and weak extreme-tail separation vary by "
        "machine ID and seed."
    )

report = f'''# Valve Low-FAR Diagnostic

## 1. Purpose

This final diagnostic asks why Valve retains moderate overall ranking while detecting
few abnormal clips at the frozen low-false-alert operating point. It distinguishes
validation-to-test threshold shift from weak separation in the extreme score tail.

## 2. Existing Frozen Scores and Thresholds

Exact score vectors were not serialized previously. They were regenerated by executing
the existing frozen pipeline exactly as written. For all 20 Valve ID/seed conditions,
the regenerated validation threshold, test-normal clip FAR, abnormal-clip detection,
and cohort counts match the frozen validation result table within `1e-12`.

## 3. Score-Tail Findings

The CSV contains count, minimum, p25, median, p75, p90, p95, p99, and maximum for
validation-normal, test-normal, and abnormal clip scores for every condition. Frozen
test false-positive and detected-abnormal counts are also retained. The p99 comparison
shows the relevant extreme tails directly; averages alone are not used to infer the
low-FAR mechanism.

## 4. Low-FPR ROC Findings

{diagnostic_markdown_table(diagnostic_summary.drop(columns="interpretation"))}

The figure shows empirical ROC curves only from 0% to 1% held-out-normal FPR. The CSV
also reports standardized partial AUROC, the maximum empirical TPR at FPR <=1%, the
largest attainable empirical FPR not exceeding 1%, and the associated thresholds.
These are post-hoc separability diagnostics, not deployable operating points.

![Valve low-FPR ROC diagnostic](valve_low_far_diagnostic.png)

## 5. Post-Hoc Test-Normal 1% Diagnostic

The post-hoc threshold uses the same order statistic and strict `score > threshold`,
but is derived from held-out test-normal clips solely for explanation. It does not
replace the frozen validation-normal threshold and must not be reported as deployment
performance.

## 6. Interpretation

{chr(10).join(interpretation_lines)}

**Cross-Valve conclusion:** {cross_valve}

## 7. Limitation

At roughly 143-200 test-normal clips, 1% FPR corresponds to only one or two false
positive clips. Every seed and exact count is therefore reported; one-clip differences
must not be over-interpreted. This analysis is post-hoc and cannot establish a new
threshold policy or deployment guarantee.

## 8. Final Experimental Closure

No model, feature, aggregation, or threshold policy was changed. This diagnostic is
explanatory and post-hoc. Research experiments are now frozen for Final Report
preparation.
'''
VALVE_DIAGNOSTIC_REPORT.write_text(report, encoding="utf-8")

required_diagnostic_outputs = [
    VALVE_DIAGNOSTIC_FIGURE,
    VALVE_DIAGNOSTIC_CSV,
    VALVE_DIAGNOSTIC_REPORT,
]
assert all(path.is_file() and path.stat().st_size > 0 for path in required_diagnostic_outputs)
saved_diagnostic = pd.read_csv(VALVE_DIAGNOSTIC_CSV)
pd.testing.assert_frame_equal(
    saved_diagnostic, valve_diagnostic, check_dtype=False, rtol=1e-12, atol=1e-12
)
published_diagnostic_text = "\n".join(
    path.read_text(encoding="utf-8", errors="ignore")
    for path in required_diagnostic_outputs if path.suffix in {".md", ".csv"}
)
assert "C:\\Users\\" not in published_diagnostic_text
assert "D:\\datasets\\" not in published_diagnostic_text

display(diagnostic_summary)
print("Recovered and verified all 20 Valve score sets against frozen results.")
print("Created exactly one diagnostic CSV, one figure, and one concise report.")
print("Research experiments are frozen. Next task: Final Report consolidation. DCASE: not opened.")


,ID,frozen threshold mean,frozen FAR mean (range),frozen detection mean (range),max TPR <=1% FPR mean (range),post-hoc TPR mean (range),interpretation
0,id_00,1.5643,0.80% (0.00%-1.51%),4.03% (0.84%-10.08%),2.86% (0.84%-5.04%),2.86% (0.84%-5.04%),B: tail-overlap limitation
1,id_02,1.4795,0.70% (0.00%-1.40%),2.33% (0.00%-4.17%),5.00% (0.00%-15.83%),5.00% (0.00%-15.83%),B: tail-overlap limitation
2,id_04,1.6371,0.50% (0.00%-2.00%),0.83% (0.00%-2.50%),3.50% (0.00%-10.00%),3.50% (0.00%-10.00%),B: tail-overlap limitation
3,id_06,1.5036,0.90% (0.50%-1.51%),2.50% (0.00%-5.00%),3.33% (0.83%-6.67%),3.33% (0.83%-6.67%),B: tail-overlap limitation


Recovered and verified all 20 Valve score sets against frozen results.
Created exactly one diagnostic CSV, one figure, and one concise report.
Research experiments are frozen. Next task: Final Report consolidation. DCASE: not opened.
